Spark Structured Streaming: Output Modes
1. What is Output Mode?
Output Mode decides which rows from the streaming result are written to the sink after each trigger/micro-batch.
There are 3 output modes:
1.	Append
2.	Update
3.	Complete
Easy way to remember
Mode	What Spark outputs
Append	Only final/new rows
Update	Only rows whose result changed
Complete	Entire result table

Important: Update mode does not mean SQL UPDATE, and Complete mode does not automatically mean overwriting a Delta table. The behavior also depends on the sink.

🟢 1. Append Mode
Definition
Append Mode writes only newly produced rows to the sink.
Previously written output is not sent again.
.writeStream
.outputMode("append")
Simple Example
Batch 1
id name age
1 Alice 30
2 Bob 25
These rows are written.
Batch 2
id name age
3 John 28
Only the new row is written:
3 John 28
Old rows are not rewritten.

What if the same record comes again?
Suppose this was already processed:
100, Alice, 30
And later the source sends the same row again:
100, Alice, 30
Without explicit deduplication, Spark can process it again.
Your sink can therefore contain:
100, Alice, 30
100, Alice, 30
Important
Append Mode itself does not perform deduplication.
If duplicates need to be removed, you need separate logic such as:
.dropDuplicates(["id"])
often together with a watermark for streaming deduplication.
Interview Answer
Append mode outputs only newly finalized/new rows and does not resend old output rows. It is commonly used when existing output does not need to change.
Remember
Append = Send new rows only

🔵 2. Update Mode
Definition
Update Mode outputs only the rows whose streaming result changed since the previous trigger.
It is especially useful for stateful aggregations such as:
•	count()
•	sum()
•	avg()
•	groupBy()
Python
.writeStream
.outputMode("update")
________________________________________
Simple Example
Suppose we're counting orders by category.
Batch 1
Input:
1, Books
2, Toys
Aggregation result:
Books 1
Toys 1
Spark outputs:
Books 1
Toys 1
Batch 2
New data:
3, Books
Now the result becomes:
Books 2
Toys 1
But in Update Mode, only the changed result needs to be emitted:
Books 2
Toys is not emitted again because its result didn't change.
________________________________________
Another Example
Existing state:
Books 1
Toys 1
New batch:
Books
Clothing
New state:
Books 2
Toys 1
Clothing 1
Update output:
Books 2
Clothing 1
Toys isn't output because it didn't change.
Interview Answer
Update mode outputs only the rows whose result changed since the previous trigger. It is commonly useful for streaming aggregations.
Remember
Update = Send changed results only


🟠 3. Complete Mode
Definition
Complete Mode outputs the entire result table after every trigger.
It is mainly relevant to aggregated streaming results.
Python
.writeStream
.outputMode("complete")
________________________________________
Simple Example
Suppose we're counting records by category.
Batch 1
Input:
1, Books
2, Toys
Full result:
Books 1
Toys 1
Batch 2
New data arrives:
3, Books
4, Clothing
Spark's complete aggregation becomes:
Books 2
Toys 1
Clothing 1
In Complete Mode, Spark outputs all three rows:
Books 2
Toys 1
Clothing 1
Even though Toys didn't change, it is still included.
Interview Answer
Complete mode outputs the entire result table after every trigger. It is useful when the consumer needs the full current aggregation result each time.
Remember
Complete = Send everything


Spark Streaming Output Modes: Simple Summary
Output Mode	What is written?	Easy meaning
Append	New/final rows only	New only
Update	New + changed results	New + Changed
Complete	Entire result table	Everything
Easy Example
Current result:

Books = 2
Toys = 1
New data: Books + Clothing
Mode	Output
Append	Only newly finalized rows, depending on query
Update	Books = 3, Clothing = 1
Complete	Books = 3, Toys = 1, Clothing = 1
Remember:
🟢 Append = New
🔵 Update = New + Changed
🟠 Complete = Full result

Simple rule
Update = New + Changed
Complete = New + Changed + Unchanged
Example
After Batch 1:
Category	Count
Books	1
Toys	1
Batch 2 source
Only a new Books record arrives:
Category
Books
Spark recalculates:
•	Books: 1 → 2
•	Toys: still 1
Complete Mode outputs
Category	Count
Books	2
Toys	1
Even though Toys = 1 is output again, Spark doesn't count Toys again.
Why? Because Toys = 1 is the aggregation result, not a new source record.
Think like this
SOURCE records        SPARK STATE        COMPLETE OUTPUT
Books                 Books = 1          Books = 1
Toys          →        Toys  = 1    →     Toys  = 1
New source:
Books                 Books = 2          Books = 2
              →        Toys  = 1    →     Toys  = 1
✅ Count increases when new source data affects the aggregation.
❌ Count does NOT increase because Complete Mode outputs the same result again.
Complete Mode = show/send the latest full result every batch, not process the target output again as new source data.






**🟢 1. Append Mode**

**Definition**
- 
- Append Mode writes only newly produced rows to the sink.
- Previously written output is not sent again.


In [0]:
.writeStream
.outputMode("append")


**🔵 2. Update Mode**

**Definition**

- Update Mode outputs only the rows whose streaming result changed since the previous trigger.
- It is especially useful for stateful aggregations such as:
- •	count()
- •	sum()
- •	avg()
- •	groupBy()


In [0]:
.writeStream
.outputMode("update")


**🟠 3. Complete Mode**


Definition

- Complete Mode outputs the entire result table after every trigger.
- It is mainly relevant to aggregated streaming results.


In [0]:
.writeStream
.outputMode("complete")
